# Fig. 3B to 3E: 3'UTR length in post-mortem ASD cortex

Plots the tables written by `fig3_cortex_tables` (run it first). Fig. 3A is a schematic.

- B: mean 3'UTR length per sample for each of the 17 cell types; Welch t-test per cell type, Benjamini-Hochberg adjusted across cell types.
- C: per-gene volcano plot for layer 2/3 excitatory neurons, as in the 2020 preprint: the y axis is the unadjusted P value of the linear model, and genes with an unadjusted P below 0.05 are coloured by local point density (the colouring does not mark significance). The variants `__adjp10` and `__adjp5` colour the genes with a Benjamini-Hochberg adjusted P below 0.1 and 0.05 instead. The x axis is on a signed square-root scale.
- D: SOX4 and NCAM2 in layer 2/3 excitatory neurons, one point per sample.
- E: Reactome gene-set enrichment for layer 2/3 excitatory neurons.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); panels are written to `M26_RESULTS_ROOT/Fig3/<panel>/`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
STYLE_DIR = f"{REPO_ROOT}/figures/style"
TABLES = Path(RESULTS_ROOT) / "Fig3" / "fig3_cortex_tables"
FIGURE = "Fig3"  # panels go to M26_RESULTS_ROOT/Fig3/<panel>/

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
ASD, CTRL = "#d73027", "#4575b4"  # as in Fig. 4
NS_GREY = "#cccccc"


def read(name: str) -> pd.DataFrame:
    return pd.read_csv(TABLES / name, sep="\t", dtype={"individual": str, "entrez": str})

## Fig. 3B: mean 3'UTR length per sample and cell type

Small points are samples (control above, ASD below the row centre); large points and bars are the group mean and standard error. Stars give the adjusted P of the per-cell-type t-test (* below 0.1, ** below 0.05).

In [ ]:
meta = read("fig3b_sample_celltype_length.tsv").dropna(subset=["utr3_length"])
tests = read("fig3b_celltype_tests.tsv").sort_values("panel_order")
order = list(tests.cell_type)  # top to bottom
ypos = {c: len(order) - 1 - i for i, c in enumerate(order)}


def stars(q: float) -> str:
    return "**" if q < 0.05 else "*" if q < 0.1 else "n.s."


rng = np.random.default_rng(0)  # vertical jitter of the sample points
fig, ax = plt.subplots(figsize=(S.THIRD_W_IN, S.THIRD_H_IN))
for y in np.arange(len(order) - 1) + 0.5:
    ax.axhline(y, color="#e6e6e6", lw=0.4, zorder=0)
for diag, off, col in [("Control", 0.17, CTRL), ("ASD", -0.17, ASD)]:
    d = meta[meta.diagnosis == diag]
    y = d.cell_type.map(ypos) + off + rng.uniform(-0.07, 0.07, len(d))
    ax.scatter(d.utr3_length, y, s=2.5, color=col, alpha=0.45, edgecolors="none", zorder=1)
    g = d.groupby("cell_type").utr3_length.agg(["mean", "sem"])
    ax.errorbar(
        g["mean"],
        g.index.map(ypos) + off,
        xerr=g["sem"],
        fmt="o",
        ms=2.6,
        color=col,
        elinewidth=0.8,
        capsize=0,
        zorder=2,
    )
ax.set_yticks([ypos[c] for c in order], order)
ax.set_ylim(-0.6, len(order) - 0.4)
ax.tick_params(axis="y", length=0)
xmax = meta.utr3_length.max()
ax.set_xlim(meta.utr3_length.min() - 20, xmax + 160)
for _, r in tests.iterrows():
    ax.text(
        xmax + 150,
        ypos[r.cell_type],
        stars(r.fdr),
        ha="right",
        va="center",
        fontsize=S.FONT_TICK if r.fdr < 0.1 else S.FONT_LEGEND,
    )
ax.set_xlabel("Mean 3'UTR length (bp)")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
handles = [
    Line2D([], [], marker="o", ls="", color=CTRL, ms=3, label="Control"),
    Line2D([], [], marker="o", ls="", color=ASD, ms=3, label="ASD"),
]
ax.legend(
    handles=handles,
    ncol=2,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    handletextpad=0.2,
    columnspacing=1.0,
)
fig.subplots_adjust(left=0.33, right=0.97, bottom=0.11, top=0.94)
S.save_and_show(fig, "fig3b_celltype_length", figure=FIGURE)
print(tests[["cell_type", "difference", "p", "fdr", "fdr_preprint"]].to_string(index=False))

## Fig. 3C: per-gene volcano plot, layer 2/3 excitatory neurons

Every tested gene is shown at its unadjusted P value. In the panel used in the figure, genes with an unadjusted P below 0.05 are coloured by local point density, as in the 2020 preprint; the panel says so, because only one of these genes passes an adjusted P of 0.05 (Benjamini-Hochberg). The two variants colour the genes below an adjusted P of 0.1 or 0.05 by direction (red, longer in ASD; blue, shorter), with the dashed line at the corresponding unadjusted P. The x axis uses a signed square-root scale.

In [ ]:
genes = read("fig3c_l23_genes.tsv")
genes["nlp"] = -np.log10(genes.p)
genes["label"] = genes.symbol.fillna(genes.entrez)

# label positions (offset in points from the gene's point)
LABEL_OFFSETS = {
    "PTPN6": (6, 3),
    "NCAM2": (7, 7),
    "FADS1": (-6, 11),
    "TTC28": (9, -8),
    "SOX4": (-8, -9),
    "BDNF": (16, 16),
}


def ssqrt(x):
    return np.sign(x) * np.sqrt(np.abs(x))


def ssqrt_inv(y):
    return np.sign(y) * y**2


def volcano(fdr_cut: float, panel: str, reference: tuple[str, ...] = ("BDNF",)) -> None:
    sig = genes.fdr < fdr_cut
    fig, ax = plt.subplots(figsize=(S.THIRD_W_IN, S.THIRD_H_IN))
    ax.set_xscale("function", functions=(ssqrt, ssqrt_inv))
    ns = genes[~sig]
    ax.scatter(ns.difference, ns.nlp, s=2, c=NS_GREY, edgecolors="none", rasterized=True)
    hit = genes[sig]
    cols = np.where(hit.difference > 0, S.COL_LONG, S.COL_SHORT)
    ax.scatter(hit.difference, hit.nlp, s=9, c=cols, edgecolors="none", zorder=3)
    thr = -np.log10(genes.loc[sig, "p"].max())
    ax.axhline(thr, ls="--", c="grey", lw=0.5, zorder=1)
    ax.axvline(0, ls=":", c="grey", lw=0.5, zorder=1)
    xlim = 1.05 * genes.difference.abs().max()
    ax.set_xlim(-xlim, xlim)
    ax.set_xticks([-2000, -500, -100, 0, 100, 500, 2000])
    ax.tick_params(axis="x", labelsize=S.FONT_TICK - 0.5)
    ax.set_ylim(0, genes.nlp.max() * 1.3)
    ax.text(
        -xlim * 0.97,
        thr,
        f"FDR {fdr_cut:.0%}",
        ha="left",
        va="bottom",
        fontsize=S.FONT_LEGEND,
        color="grey",
    )
    ax.text(
        0.03,
        0.98,
        "shorter\nin ASD",
        transform=ax.transAxes,
        ha="left",
        va="top",
        color=S.COL_SHORT,
        fontsize=S.FONT_ANNOT,
    )
    ax.text(
        0.97,
        0.98,
        "longer\nin ASD",
        transform=ax.transAxes,
        ha="right",
        va="top",
        color=S.COL_LONG,
        fontsize=S.FONT_ANNOT,
    )
    lab = pd.concat([hit, genes[genes.symbol.isin(reference) & ~sig]])
    for _, r in lab.iterrows():
        dx, dy = LABEL_OFFSETS.get(r.label, (5, 4))
        ax.annotate(
            r.label,
            (r.difference, r.nlp),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=S.FONT_ANNOT - 0.5,
            fontstyle="italic",
            ha="left" if dx >= 0 else "right",
            va="center",
            color="black" if r.fdr < fdr_cut else "dimgrey",
            arrowprops=dict(arrowstyle="-", color="grey", lw=0.3, shrinkA=0, shrinkB=1.5),
        )
    ax.set_xlabel("3'UTR length difference (bp)\nASD minus control")
    ax.set_ylabel(r"$-\log_{10}\,P$")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.17, top=0.97)
    S.save_and_show(fig, panel, figure=FIGURE)
    n_long, n_short = int((hit.difference > 0).sum()), int((hit.difference < 0).sum())
    print(f"{panel}: {len(genes)} genes, FDR < {fdr_cut}: {n_long} longer, {n_short} shorter")
    shown = ["label", "difference", "p", "fdr", "n_samples"]
    print(hit.sort_values("p")[shown].to_string(index=False))


volcano(0.10, "fig3c_l23_volcano__adjp10")
volcano(0.05, "fig3c_l23_volcano__adjp5")


# panel of the figure: genes with unadjusted P < 0.05 coloured by point density, as in 2020
DENSITY_CMAP = LinearSegmentedColormap.from_list(
    "density", ["#104e8b", "#1e90ff", "#ffa500", "#ff4500", "#ff0000", "#ff0000"]
)
RAW_OFFSETS = {"SOX4": (-10, 9), "NCAM2": (8, 9), "BDNF": (16, 10)}


def volcano_raw(panel: str, labelled: tuple[str, ...] = ("SOX4", "NCAM2", "BDNF")) -> None:
    raw = genes.p < 0.05
    fig, ax = plt.subplots(figsize=(S.THIRD_W_IN, S.THIRD_H_IN))
    ax.set_xscale("function", functions=(ssqrt, ssqrt_inv))
    ns = genes[~raw]
    ax.scatter(ns.difference, ns.nlp, s=2, c=NS_GREY, edgecolors="none", rasterized=True)
    hit = genes[raw]
    xy = np.vstack(
        [ssqrt(hit.difference) / ssqrt(genes.difference.abs().max()), hit.nlp / genes.nlp.max()]
    )
    dens = gaussian_kde(xy, bw_method=0.1)(xy)
    o = np.argsort(dens)
    pts = ax.scatter(
        hit.difference.iloc[o],
        hit.nlp.iloc[o],
        s=3,
        c=dens[o],
        cmap=DENSITY_CMAP,
        edgecolors="none",
        rasterized=True,
    )
    ax.axhline(-np.log10(0.05), ls="-", c="#e41a1c", alpha=0.4, lw=0.4, zorder=1)
    ax.axvline(0, ls=":", c="grey", lw=0.5, zorder=1)
    xlim = 1.05 * genes.difference.abs().max()
    ax.set_xlim(-xlim, xlim)
    ax.set_xticks([-2000, -500, -100, 0, 100, 500, 2000])
    ax.tick_params(axis="x", labelsize=S.FONT_TICK - 0.5)
    ax.set_ylim(0, genes.nlp.max() * 1.3)
    ax.text(
        0.03,
        0.98,
        "shorter\nin ASD",
        transform=ax.transAxes,
        ha="left",
        va="top",
        color=S.COL_SHORT,
        fontsize=S.FONT_ANNOT,
    )
    ax.text(
        0.97,
        0.98,
        "longer\nin ASD",
        transform=ax.transAxes,
        ha="right",
        va="top",
        color=S.COL_LONG,
        fontsize=S.FONT_ANNOT,
    )
    cax = ax.inset_axes([0.05, 0.83, 0.3, 0.022])
    cb = fig.colorbar(pts, cax=cax, orientation="horizontal", ticks=[])
    cb.outline.set_linewidth(0.3)
    cax.set_title("point density", fontsize=S.FONT_LEGEND, pad=1.5)
    ax.text(
        0.05,
        0.805,
        "colour: unadjusted\nP < 0.05 (not corrected\nfor multiple testing)",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=S.FONT_LEGEND - 0.5,
        color="#e41a1c",
    )
    for _, r in genes[genes.symbol.isin(labelled)].iterrows():
        dx, dy = RAW_OFFSETS.get(r.label, (5, 4))
        ax.annotate(
            r.label,
            (r.difference, r.nlp),
            xytext=(dx, dy),
            textcoords="offset points",
            fontsize=S.FONT_ANNOT - 0.5,
            fontstyle="italic",
            ha="left" if dx >= 0 else "right",
            va="center",
            arrowprops=dict(arrowstyle="-", color="grey", lw=0.3, shrinkA=0, shrinkB=1.5),
        )
    ax.set_xlabel("3'UTR length difference (bp)\nASD minus control")
    ax.set_ylabel(r"$-\log_{10}\,P$")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    fig.subplots_adjust(left=0.16, right=0.96, bottom=0.17, top=0.97)
    S.save_and_show(fig, panel, figure=FIGURE)
    longer, shorter = int((hit.difference > 0).sum()), int((hit.difference < 0).sum())
    adj = genes.fdr
    print(f"{panel}: {len(genes)} genes; unadjusted P < 0.05: {longer} longer, {shorter} shorter;")
    print(f"adjusted P < 0.05: {int((adj < 0.05).sum())}, < 0.1: {int((adj < 0.1).sum())}")


volcano_raw("fig3c_l23_volcano")

## Fig. 3D: SOX4 and NCAM2 in layer 2/3 excitatory neurons

One point per sample; black bars are the group mean and standard error.

In [ ]:
ex = read("fig3d_l23_examples.tsv")
ex["diagnosis"] = pd.Categorical(ex.diagnosis, ["Control", "ASD"])
fig, axes = plt.subplots(1, 2, figsize=(S.THIRD_W_IN, S.QUARTER_H_IN))
for ax, gene in zip(axes, ["SOX4", "NCAM2"]):
    d = ex[ex.gene == gene]
    sns.swarmplot(
        data=d,
        x="diagnosis",
        y="utr3_length",
        hue="diagnosis",
        palette={"Control": CTRL, "ASD": ASD},
        size=2.2,
        ax=ax,
        legend=False,
    )
    g = d.groupby("diagnosis", observed=True).utr3_length.agg(["mean", "sem"])
    for i, (m, se) in enumerate(zip(g["mean"], g["sem"])):
        ax.plot([i - 0.25, i + 0.25], [m, m], color="black", lw=0.9, zorder=3)
        ax.plot([i, i], [m - se, m + se], color="black", lw=0.9, zorder=3)
    ax.set_title(gene, fontstyle="italic", fontsize=S.FONT_LABEL, pad=3)
    ax.set_xlabel("")
    ax.set_ylabel("")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylabel("Mean 3'UTR length (bp)")
fig.subplots_adjust(left=0.2, right=0.98, bottom=0.12, top=0.9, wspace=0.55)
S.save_and_show(fig, "fig3d_l23_examples", figure=FIGURE)

## Fig. 3E: Reactome gene-set enrichment, layer 2/3 excitatory neurons

Pathways with an adjusted P below 0.1, ordered by normalised enrichment score (NES); bar length is the number of genes of the pathway in the ranking, colour is the NES.

In [ ]:
gsea = read("fig3e_l23_reactome_gsea.tsv").sort_values("NES")
cmap = LinearSegmentedColormap.from_list("nes", ["#9ecae1", "#08306b"])
norm = Normalize(gsea.NES.min(), gsea.NES.max())
fig, ax = plt.subplots(figsize=(S.TWOTHIRD_W_IN, S.QUARTER_H_IN))
ax.barh(gsea.Description, gsea.setSize, color=cmap(norm(gsea.NES)), height=0.7)
ax.set_xlabel("Genes in pathway")
ax.tick_params(axis="y", length=0)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
cb = fig.colorbar(sm, ax=ax, fraction=0.04, pad=0.03, aspect=12)
cb.set_label("NES", fontsize=S.FONT_LEGEND_TITLE)
cb.ax.tick_params(labelsize=S.FONT_LEGEND, length=2)
cb.outline.set_linewidth(0.4)
fig.subplots_adjust(left=0.42, right=0.9, bottom=0.18, top=0.96)
S.save_and_show(fig, "fig3e_l23_reactome_gsea", figure=FIGURE)
print(gsea[["Description", "setSize", "NES", "p.adjust"]].to_string(index=False))